In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import os

# ============================================
# MODEL DEFINITION (YOUR EXACT MODEL.PY)
# ============================================

class EfficientNetEncoder(nn.Module):
    def __init__(self, model_name='efficientnet_b3', pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        self.feature_channels = self.backbone.feature_info.channels()
    
    def forward(self, x):
        return self.backbone(x)


class SegFormerDecoder(nn.Module):
    def __init__(self, encoder_channels, num_classes=1, decoder_dim=256):
        super().__init__()
        self.proj = nn.ModuleList()
        for in_ch in encoder_channels:
            self.proj.append(
                nn.Sequential(
                    nn.Conv2d(in_ch, decoder_dim, kernel_size=1),
                    nn.BatchNorm2d(decoder_dim),
                    nn.ReLU(inplace=True)
                )
            )
        self.fusion = nn.Sequential(
            nn.Conv2d(decoder_dim * len(encoder_channels), decoder_dim, kernel_size=1),
            nn.BatchNorm2d(decoder_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(decoder_dim, decoder_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(decoder_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(decoder_dim, num_classes, kernel_size=1)
        )
    
    def forward(self, features):
        target_size = features[0].shape[2]
        projected = []
        for i, feat in enumerate(features):
            proj = self.proj[i](feat)
            if proj.shape[2] != target_size:
                proj = F.interpolate(proj, size=(target_size, target_size), 
                                     mode='bilinear', align_corners=False)
            projected.append(proj)
        concat = torch.cat(projected, dim=1)
        out = self.fusion(concat)
        out = F.interpolate(out, size=(224, 224), mode='bilinear', align_corners=False)
        return out


class LungUltrasoundModel(nn.Module):
    def __init__(self, num_classes=3, num_seg_classes=1):
        super().__init__()
        self.encoder = EfficientNetEncoder('efficientnet_b3', pretrained=False)
        encoder_channels = self.encoder.feature_channels
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(encoder_channels[-1], 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
        self.seg_decoder = SegFormerDecoder(encoder_channels, num_seg_classes, 256)
    
    def forward(self, x):
        features = self.encoder(x)
        class_out = self.classifier(features[-1])
        seg_out = self.seg_decoder(features)
        return class_out, seg_out


# ============================================
# CONVERT YOUR TRAINED MODEL
# ============================================

print("="*60)
print("CONVERTING TRAINED MODEL TO PYTORCH_MODEL.BIN")
print("="*60)

# Step 1: Create the model architecture
model = LungUltrasoundModel(num_classes=3, num_seg_classes=1)
print("✅ Model architecture created")

# Step 2: Specify the path to your trained model
# UPDATE THIS PATH to where your model5_best_acc.pth is located
MODEL_SOURCE_PATH = "./models/model5_best_dice.pth"  # <-- CHANGE THIS IF NEEDED

# Check if the file exists
if not os.path.exists(MODEL_SOURCE_PATH):
    print(f"\n❌ Model not found at: {MODEL_SOURCE_PATH}")
    print("\nSearching for model files in common locations...")
    
    # Search for model files
    search_paths = [
        "./models/model5_best_acc.pth",
        "./model5_best_acc.pth",
        "../models/model5_best_acc.pth",
        "models/model5_best_acc.pth",
        r"C:/Users/Administrator/Documents/LUS-Research/models/model5_best_acc.pth",
    ]
    
    found = False
    for path in search_paths:
        if os.path.exists(path):
            MODEL_SOURCE_PATH = path
            found = True
            print(f"✅ Found model at: {MODEL_SOURCE_PATH}")
            break
    
    if not found:
        print("\n❌ Could not find model file. Please update MODEL_SOURCE_PATH")
        exit()

# Step 3: Load the trained weights
print(f"\n📂 Loading weights from: {MODEL_SOURCE_PATH}")
print(f"   File size: {os.path.getsize(MODEL_SOURCE_PATH) / 1024 / 1024:.2f} MB")

state_dict = torch.load(MODEL_SOURCE_PATH, map_location='cpu')
model.load_state_dict(state_dict, strict=False)
model.eval()
print("✅ Weights loaded successfully")

# Step 4: Save as pytorch_model.bin
OUTPUT_PATH = "pytorch_model.bin"
torch.save(model.state_dict(), OUTPUT_PATH)

output_size = os.path.getsize(OUTPUT_PATH) / 1024 / 1024
print(f"\n✅ Model saved as: {OUTPUT_PATH}")
print(f"   File size: {output_size:.2f} MB")
print(f"   Location: {os.path.abspath(OUTPUT_PATH)}")

print("\n" + "="*60)
print("📤 READY TO UPLOAD TO GOOGLE DRIVE!")
print("="*60)
print("\nNext steps:")
print("1. Go to https://drive.google.com")
print("2. Upload pytorch_model.bin")
print("3. Right-click → Share → 'Anyone with the link'")
print("4. Copy the File ID from the URL")
print("5. Update FILE_ID in your streamlit_app.py")

CONVERTING TRAINED MODEL TO PYTORCH_MODEL.BIN
✅ Model architecture created

📂 Loading weights from: ./models/model5_best_dice.pth
   File size: 44.48 MB
✅ Weights loaded successfully

✅ Model saved as: pytorch_model.bin
   File size: 44.49 MB
   Location: C:\Users\Administrator\pytorch_model.bin

📤 READY TO UPLOAD TO GOOGLE DRIVE!

Next steps:
1. Go to https://drive.google.com
2. Upload pytorch_model.bin
3. Right-click → Share → 'Anyone with the link'
4. Copy the File ID from the URL
5. Update FILE_ID in your streamlit_app.py
